# Calidad de evidencia de Jupiter Price V3
## tl;dr
El contrato distingue precio valido (OK), omision documentada (NIL) y fallo desconocido (ERR). Ninguno demuestra una ruta ejecutable. Este cuaderno verifica 12 casos sinteticos, no el rendimiento del bot ni respuestas actuales del proveedor.


## Context & Methods
Grano: un punto por mint solicitado dentro de una respuesta V3. La hora usada es un recibo HTTP ficticio, no la hora del ultimo swap. Contrato ejecutado: `fetcher/jupiter_price_v3.py`. Pruebas de transporte y consumidores: `tests/test_jupiter_price_evidence.py`.
### Key Assumptions
La API V3 devuelve un mapa directo mint -> punto; `usdPrice` es el unico precio. Solo una clave ausente en un mapa valido cuenta como omision. `blockId` y `decimals` se conservan cuando existen; no se deduce recencia del mercado sin otra fuente. No hay peticiones HTTP, SQL del operador, wallet, entrenamientos ni ordenes. Una tabla basta para estos casos de contrato; no representa una distribucion de mercado.
Fuentes consultadas el 2026-10-08: [Jupiter Price](https://developers.jup.ag/docs/price), [Referencia V3](https://developers.jup.ag/docs/api-reference/price).


In [1]:
from pathlib import Path
import hashlib, inspect, json, sys
import pandas as pd
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'fetcher' / 'jupiter_price_v3.py').exists())
sys.path.insert(0, str(root))
from fetcher import jupiter_price_v3 as contract
print('Contrato SHA256 (fuente normalizada):', hashlib.sha256(inspect.getsource(contract).encode()).hexdigest())


Contrato SHA256 (fuente normalizada): fa86f1f404d0837a352cea65c5d3f8ca285b97850041d7d7806ae686c4630202


## Data
Fixtures sinteticos autocontenidos. Precio: USD por token; blockId: identificador de bloque, no segundos. Los casos no son trades.


In [2]:
mint = 'JUPyiwrYJFskUPiHa7hkeR8VUtAeFoSYbKedZNsDvCN'
other = 'EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v'
good = {'usdPrice': 2.0, 'blockId': 123, 'decimals': 6}
def encoded(payload): return json.dumps(payload).encode()
cases = [
 ('Precio valido', encoded({mint: good}), 'OK'),
 ('Omision oficial', encoded({}), 'NIL'),
 ('Envelope de error', encoded({'error': 'unavailable'}), 'ERR'),
 ('Wrapper legado', encoded({'data': {mint: good}}), 'ERR'),
 ('Punto null', encoded({mint: None}), 'ERR'),
 ('Precio booleano', encoded({mint: {'usdPrice': True}}), 'ERR'),
 ('Precio string', encoded({mint: {'usdPrice': '2'}}), 'ERR'),
 ('Metadata invalida', encoded({mint: {**good, 'decimals': 256}}), 'ERR'),
 ('Mint no solicitado', encoded({other: good}), 'ERR'),
 ('Campo duplicado', ('{"' + mint + '":{"usdPrice":2,"usdPrice":3}}').encode(), 'ERR'),
 ('JSON truncado', b'{', 'ERR'),
 ('Cuerpo > 1 MiB', b' ' * (contract.MAX_PRICE_BODY_BYTES + 1), 'ERR'),
]


## Results
Comparacion con el resultado esperado del contrato; no es una estimacion de precision sobre datos reales.


In [3]:
rows = []
for label, body, expected in cases:
    batch = contract.decode_price_body(body, [mint], received_at=100.0)
    point = batch.points[mint]
    rows.append({'Caso': label, 'Esperado': expected, 'Observado': point.status, 'Motivo': point.reason})
result = pd.DataFrame(rows)
assert (result['Esperado'] == result['Observado']).all()
assert len(result) == 12
display(result)


,Caso,Esperado,Observado,Motivo
0,Precio valido,OK,OK,provider_price
1,Omision oficial,NIL,NIL,provider_omitted_price
2,Envelope de error,ERR,ERR,invalid_response_map
3,Wrapper legado,ERR,ERR,invalid_response_map
4,Punto null,ERR,ERR,invalid_price_point
5,Precio booleano,ERR,ERR,invalid_price_point
6,Precio string,ERR,ERR,invalid_price_point
7,Metadata invalida,ERR,ERR,invalid_price_metadata
8,Mint no solicitado,ERR,ERR,invalid_response_map
9,Campo duplicado,ERR,ERR,invalid_json_body


In [4]:
point = contract.decode_price_body(encoded({mint: good}), [mint], received_at=100.0).points[mint]
assert point.price_usd == 2.0 and point.block_id == 123 and point.decimals == 6
assert 'has_route' not in point.__dict__
print('12/12 casos de contrato coherentes; metadatos conservados; sin evidencia de ruta ni rentabilidad.')


12/12 casos de contrato coherentes; metadatos conservados; sin evidencia de ruta ni rentabilidad.


## Takeaways
Los errores no deben envenenar la cache negativa ni transformarse en etiquetas de oportunidades perdidas. Una omision de precio no explica por si sola por que el bot no compro. La verificacion de liquidez/cantidad requiere cotizaciones separadas. Estos casos prueban distinciones de software; faltan respuestas actuales, operaciones cerradas con costes y evidencia prospectiva para afirmar beneficios. El bot debe permanecer apagado durante esta auditoria.
